In [1]:
import copy
import gc
import json
import math
import numpy as np
import os
import pandas as pd
import requests

from scipy.spatial.distance import cosine, euclidean

import IPython.core.display


In [2]:
# Get the root_path for this jupyter notebook repo.
repo_path = os.path.dirname(os.path.abspath(os.getcwd()))
pc_data_path = os.path.join(
    repo_path, 'files', 'poggio-civitate',
)
# this is the path to the main dataset, with all of the cataloged objects.
object_data_path = os.path.join(
    pc_data_path, 'pc-catalog-objs-embeddings.csv',
)
output_html = os.path.join(
    pc_data_path, 'pc-catalog-objs-embeddings.html',
)
MAX_RECORDS = 1500
KEYWORD = 'Bucchero Diag'
TITLE = 'Poggio Civitate Cataloged Diagnostic Bucchero'
file_keyword = str(KEYWORD).replace(' ', '-').lower()
file_suffix = '-multi'

sampled_csv =  os.path.join(
    pc_data_path, f'pc-catalog-objs-sampled-{file_keyword}{file_suffix}.csv',
)


distances_csv =  os.path.join(
    pc_data_path, f'pc-catalog-objs-distance-{file_keyword}{file_suffix}.csv',
)


net_output_html = os.path.join(
    pc_data_path, f'pc-catalog-objs-embeddings-{file_keyword}{file_suffix}.html',
)


df_all = pd.read_csv(object_data_path)
print(f'We have {len(df_all.index)} records of object data')
print(df_all.columns.tolist())

We have 13541 records of object data
['Unnamed: 0', 'uuid', 'label', 'path', 'item_class__slug', 'project__label', 'project__uuid', 'geo_source__path', 'geo_source__uuid', 'chrono_source__path', 'chrono_source__uuid', 'latitude', 'longitude', 'earliest', 'latest', 'str_for_embedding', 'embedding', 'Unnamed: 0.5', 'Unnamed: 0.4', 'Unnamed: 0.3', 'Unnamed: 0.2', 'Unnamed: 0.1', 'thumbnail_uri']


In [3]:
def get_uuids_by_query(url):
    if not url:
        return None
    if '#' in url:
        url = url.split('#')[0]
    if not 'response=uuid' in url:
        url += '&response=uuid'
    if not 'rows=' in url:
        url += '&rows=2000'
    headers = {
        'User-Agent': 'oc-api-client',
        'Accept': 'application/json', 
    }
    r = requests.get(url, headers=headers)
    uuids = r.json()
    return uuids

url = None 
# bucchero vessels
# url = 'https://opencontext.org/query/Europe/Italy.json?response=uuid&rows=2000&cat=oc-gen-cat-object&proj=24-murlo&prop=24-fabric-category---24-bucchero&prop=24-object-type---24-vessel&prop=24-vessel-form&type=subjects'    
# Bowls
# url = 'https://opencontext.org/query/Europe/Italy?cat=oc-gen-cat-object&images=1&proj=24-murlo&prop=24-object-type---24-vessel&prop=24-vessel-form---24-bowl&type=subjects#tab=3/aq=related-media---undefined/ovgrd=oc/zm=15/ov=sqr'
# Diagnostic bucchero
url = 'https://opencontext.org/query/Europe/Italy?cat=oc-gen-cat-object&proj=24-murlo&prop=24-fabric-category---24-bucchero&prop=24-object-type---24-vessel&prop=24-vessel-form---24-bowl%7C%7C24-cup%7C%7C24-lid%7C%7C24-plate%7C%7C24-vase%7C%7C24-kantharos%7C%7C24-oinochoe%7C%7C24-tondo%7C%7C24-dish%7C%7C24-tripod%7C%7C24-skyphos%7C%7C24-cooking-stand%7C%7C24-alabastron%7C%7C24-jar%7C%7C24-jug%7C%7C24-krater%7C%7C24-kylix%7C%7C24-lekythos%7C%7C24-omphalos&type=subjects'
# Rocchetti
# url = 'https://opencontext.org/query/Europe/Italy?cat=oc-gen-cat-object&proj=24-murlo&prop=24-object-type---24-textile-related---24-textile-relatedrocchetto&type=subjects'

uuids = None
uuids = get_uuids_by_query(url)
if uuids:
    print(f'UUIDs to process: {len(uuids)}')

UUIDs to process: 383


In [4]:
def get_valid_data_sample(df_all, keyword=KEYWORD, uuids=None, n=MAX_RECORDS, filepath=sampled_csv):
    if filepath and os.path.exists(filepath):
        print(f'Reading previously cached sample data from {filepath}')
        df_sample = pd.read_csv(filepath)
        df_sample['embedding'] = df_sample['embedding'].apply(lambda x: json.loads(x))
        return df_sample
    valid_index = (
        ~df_all['embedding'].isnull()
        & ~df_all['latitude'].isnull()
        & ~df_all['longitude'].isnull()
        & ~df_all['uuid'].isnull()
    )
    if keyword and not uuids:
        valid_index &= df_all['str_for_embedding'].str.contains(keyword)
    if uuids and isinstance(uuids, list):
        valid_index &= df_all['uuid'].isin(uuids)
    if len(df_all[valid_index].index) < n:
        n = len(df_all[valid_index].index)
    df_sample = df_all[valid_index].sample(n=n, random_state=1).copy()
    df_sample.reset_index(drop=True, inplace=True)
    if filepath:
        df_sample.to_csv(filepath, index=False)
    df_sample['embedding'] = df_sample['embedding'].apply(lambda x: json.loads(x))
    return df_sample

df_sample = get_valid_data_sample(df_all, keyword=KEYWORD, uuids=uuids, n=MAX_RECORDS)
print(f'We have {len(df_sample.index)} valid sample records of object data')

Reading previously cached sample data from /home/ekansa/github/open-context-jupyter/files/poggio-civitate/pc-catalog-objs-sampled-bucchero-diag-multi.csv
We have 383 valid sample records of object data


In [5]:
# Release some memory
del df_all
gc.collect()

10

In [6]:
def get_difference_between_embeddings(e1, e2):
    """Get a difference distance between two embeddings"""
    v1 = np.array(e1)
    v2 = np.array(e2)
    return cosine(v1, v2)

def get_geodist(lat_1, lon_1, lat_2, lon_2, max_dist=None):
    lat_sqr = (lat_1 - lat_2)**2
    lon_sqr = (lon_1 - lon_2)**2
    dist = math.sqrt((lat_sqr + lon_sqr))
    if not max_dist:
        return dist
    # Return the distance, normalized as a fraction of the
    # max_distance
    return dist / max_dist


In [7]:
from itertools import combinations

def make_raw_distance_matrix(df_sample, filepath=distances_csv):
    if os.path.exists(filepath):
        # We have already cacheded the distance data, no need to recalcuate
        df_raw_dist = pd.read_csv(filepath)
        return df_raw_dist
    # make the maximum geo distance to normalize the geo distances
    max_lat = df_sample['latitude'].max()
    max_lon = df_sample['longitude'].max()
    min_lat = df_sample['latitude'].min()
    min_lon = df_sample['longitude'].min()
    max_geo_dist = get_geodist(lat_1=max_lat, lon_1=max_lon, lat_2=min_lat, lon_2=min_lon)
    count_sample = len(df_sample.index)

    # get cols for columns relating to controlled vocab predicates
    cvocab_cols = [c for c in df_sample.columns.tolist() if c.startswith('pred_') and c.endswith('__v')]
    print(f'Prepare arrays for: {cvocab_cols}')
    for cvocab_col in cvocab_cols:
        df_sample[cvocab_col] = df_sample[cvocab_col].apply(json.loads)
        df_sample[cvocab_col] = df_sample[cvocab_col].apply(np.array)

    df_sample['embedding'] = df_sample['embedding'].apply(np.array)
    df_sample['uuid_g'] = df_sample['uuid'].str.replace('-', '_')
    uuid_g_list = df_sample['uuid_g'].unique().tolist()
    # make unique combinations of all sources and target uuid_g
    rows = [{'source': a, 'target': b,} for a, b in combinations(uuid_g_list, 2)]
    df_raw_dist = pd.DataFrame(data=rows)

    print('generated source, target dataframe...')

    join_cols = ['uuid_g', 'uuid', 'label', 'latitude', 'longitude', 'embedding'] + cvocab_cols
    df_raw_dist = df_raw_dist.merge(df_sample[join_cols], left_on='source', right_on='uuid_g', )
    df_raw_dist = df_raw_dist.merge(df_sample[join_cols], left_on='target', right_on='uuid_g', suffixes=('_sub', '_trg'))

    df_raw_dist['geo_dist'] = np.sqrt( 
        (df_raw_dist['latitude_sub'] - df_raw_dist['latitude_trg'])**2 
        + (df_raw_dist['longitude_sub']-df_raw_dist['longitude_trg'])**2
    )
    df_raw_dist['geo_dist_normal'] = df_raw_dist['geo_dist'] / max_geo_dist

    for cvocab_col in cvocab_cols:
        col_dist = cvocab_col + '_dist'
        col_dist_norm = col_dist + '_normal'
        print(f'Calculating euclidean distance for: {col_dist}')
        df_raw_dist[col_dist] = df_raw_dist.apply(
            lambda x: euclidean(x[f'{cvocab_col}_sub'], x[f'{cvocab_col}_trg']), 
            axis=1
        )
        col_max = df_raw_dist[col_dist].max()
        df_raw_dist[col_dist_norm] = df_raw_dist[col_dist] / col_max
        
    print('finished euclidean distances...')
    df_raw_dist['vector_dist'] = df_raw_dist.apply(
        lambda x: cosine(x['embedding_sub'], x['embedding_trg']), 
        axis=1
    )
    print('finished embedding cosine similarities...')
    df_raw_dist['vector_sim'] = 1 - df_raw_dist['vector_dist']
    max_vector_dist =  df_raw_dist['vector_dist'].max()
    df_raw_dist['vector_dist_normal'] = df_raw_dist['vector_dist'] / max_vector_dist
    df_raw_dist.drop_duplicates(subset=['source', 'target'], inplace=True)

    drop_cvocb_cols = [f'{c}_sub' for c in cvocab_cols]
    drop_cvocb_cols += [f'{c}_trg' for c in cvocab_cols]
    df_raw_dist.drop(columns=(['embedding_sub', 'embedding_trg'] +  drop_cvocb_cols), inplace=True)
    df_raw_dist.to_csv(filepath, index=False)
    return df_raw_dist
            
# Make raw distance dataframe
df_raw_dist = make_raw_distance_matrix(df_sample)
# df_raw_dist.head(10)
            

In [20]:
distance_weights = {
    'vector_dist_normal': 0.39,
    'geo_dist_normal': 0.01,
    'pred__24_object_type__v_dist_normal': 0.1,
    'pred__24_vessel_form__v_dist_normal': 0.1,
    'pred__24_motif__v_dist_normal': 0.1,
    'pred__24_fabric_category__v_dist_normal': 0.1,
    'pred__24_decorative_technique__v_dist_normal': 0.1,
    'pred__24_vessel_part_present__v_dist_normal': 0.1,
}

def make_weighted_similaries(df_raw_dist, distance_weights=distance_weights):
    df_sim = df_raw_dist.copy()
    cols = df_sim.columns.tolist()
    use_distance_weights = {k:v for k, v in distance_weights.items() if k in cols}
    df_sim['distance'] = 0.0
    for col, weight in use_distance_weights.items():
        print(f'add {col}, {weight}')
        # 1. Safely convert to float, turning unparseable text into NaN
        s = pd.to_numeric(df_sim[col], errors='coerce')
        # 2. Fill missing/NaN values with 0.0 so addition doesn't fail
        s = s.fillna(0.0)
        # 3. Add to total distance
        df_sim['distance'] += s * float(weight)
    df_sim['weight'] = (1 - df_sim['distance']) * 100
    return df_sim

# Get the count of the total sample
count_sample = len(df_sample.index)
# Make raw distance dataframe
# Make weighted similarities (similarity = 1 - distance)
df_sim = make_weighted_similaries(df_raw_dist, distance_weights=distance_weights)
df_sim.head(5)

,source,target,uuid_g_sub,uuid_sub,label_sub,latitude_sub,longitude_sub,uuid_g_trg,uuid_trg,label_trg,latitude_trg,longitude_trg,geo_normal_dist,vector_dist,vector_sim,vector_normal_dist,weight
0,7d6c0731_95ec_4734_2a32_abce101340e5,172822c0_4029_4efb_7bdd_89ee393d649f,7d6c0731_95ec_4734_2a32_abce101340e5,7d6c0731-95ec-4734-2a32-abce101340e5,PC 19730317,43.153527,11.401774,172822c0_4029_4efb_7bdd_89ee393d649f,172822c0-4029-4efb-7bdd-89ee393d649f,PC 20050126,43.152750,11.401812,0.147952,0.198443,0.801557,0.364420,63.666272
1,7d6c0731_95ec_4734_2a32_abce101340e5,a70a9c9a_290f_4ea6_029d_1b9f83d07d92,7d6c0731_95ec_4734_2a32_abce101340e5,7d6c0731-95ec-4734-2a32-abce101340e5,PC 19730317,43.153527,11.401774,a70a9c9a_290f_4ea6_029d_1b9f83d07d92,a70a9c9a-290f-4ea6-029d-1b9f83d07d92,PC 19720371,43.153427,11.401469,0.060962,0.251217,0.748783,0.461333,54.066897
2,7d6c0731_95ec_4734_2a32_abce101340e5,867b4e19_c7a2_4d37_34f9_770b31ac7295,7d6c0731_95ec_4734_2a32_abce101340e5,7d6c0731-95ec-4734-2a32-abce101340e5,PC 19730317,43.153527,11.401774,867b4e19_c7a2_4d37_34f9_770b31ac7295,867b4e19-c7a2-4d37-34f9-770b31ac7295,PC 19690142,43.153298,11.402240,0.098651,0.265202,0.734798,0.487015,51.492683
3,7d6c0731_95ec_4734_2a32_abce101340e5,5f2b92d5_3659_4b84_5375_9968d2f18dbd,7d6c0731_95ec_4734_2a32_abce101340e5,7d6c0731-95ec-4734-2a32-abce101340e5,PC 19730317,43.153527,11.401774,5f2b92d5_3659_4b84_5375_9968d2f18dbd,5f2b92d5-3659-4b84-5375-9968d2f18dbd,PC 19720360,43.153385,11.401445,0.068194,0.257711,0.742289,0.473258,52.876739
4,7d6c0731_95ec_4734_2a32_abce101340e5,10a2b6c9_f7e5_40c1_eee7_7a07c93ebd7c,7d6c0731_95ec_4734_2a32_abce101340e5,7d6c0731-95ec-4734-2a32-abce101340e5,PC 19730317,43.153527,11.401774,10a2b6c9_f7e5_40c1_eee7_7a07c93ebd7c,10a2b6c9-f7e5-40c1-eee7-7a07c93ebd7c,PC 19730292,43.153527,11.401774,0.000000,0.264012,0.735988,0.484829,51.759553


In [21]:

def percentile_df_sim_weights(df_sim):
    """Makes percentiles for df_sim weights for each source"""
    df_sim['source_max_weight'] = float(0.0)
    df_sim['source_min_weight'] = float(0.0)
    quantiles = [0.98, 0.9, .75, 0.5, .25, 0.1,]
    q_cols = {f'source_{int(q * 100)}_weight': q for q in quantiles}
    for col, _ in q_cols.items():
        df_sim[col] = float(0.0)
    for source in df_sim['source'].unique():
        act_index = df_sim['source'] == source
        df_sim.loc[act_index, 'source_max_weight'] = df_sim[act_index]['weight'].max()
        df_sim.loc[act_index, 'source_min_weight'] = df_sim[act_index]['weight'].min()
        for col, q in q_cols.items():
            df_sim.loc[act_index, col] = df_sim[act_index]['weight'].quantile(q)
    return df_sim

def select_min_weights_by_col(df_sim, col_criteria):
    df_sim['use_weight'] = False
    for i, row in df_sim.iterrows():
        df_sim.at[i, 'use_weight'] = (row['weight'] >= row[col_criteria])
    return df_sim

df_sim = percentile_df_sim_weights(df_sim)
# df_sim = select_min_weights_by_col(df_sim, col_criteria='source_90_weight')
df_sim = select_min_weights_by_col(df_sim, col_criteria='source_max_weight')


In [22]:
#act_index = df_sim['weight'] >= df_sim['source_98_weight']
q_top = df_sim['weight'].quantile(0.99)
act_index = df_sim['weight'] >= df_sim['source_max_weight']
act_index |= df_sim['weight'] >= q_top
print(f'Make force directed graph with {len(df_sim[act_index].index)} edges')

Make force directed graph with 963 edges


In [23]:
# Make options based on the size of the total number of nodes

iterations = 750 - round((count_sample * 2.5), 0)
if iterations < 75:
    iterations = 75

iterations = 50
options = (
"""
{
  "physics": {
    "solver": "forceAtlas2Based",
    "forceAtlas2Based": {
      "gravitationalConstant": -175,
      "centralGravity": 0.01,
      "springLength": 100,
      "springConstant": 0.08,
      "damping": 0.6
    },
    "maxVelocity": 50,
    "minVelocity": 0.75,
    "timestep": 0.5,
    "stabilization": {
      "enabled": true,
      "iterations": XX,
      "updateInterval": 25,
      "onlyDynamicEdges": false,
      "fit": true
    }
  },
  "edges": {
    "smooth": false
  },
  "interaction": {
    "hideEdgesOnDrag": true
  }
}
"""
)
options = options.replace('"iterations": XX,', f'"iterations": {iterations},')

In [24]:
from pyvis.network import Network
import networkx as nx

# 1. Create the network for the force-directed graph

net = Network(
    height='750px', 
    width='100%', 
    bgcolor='#222222', 
    font_color='white', 
    notebook=True, 
    cdn_resources='in_line'
)

# 2. Set physics options, dymanically scaled with the number of nodes
net.set_options(options)


# 3. Add unique nodes with image properties
all_nodes = df_sim[act_index]['source'].unique().tolist()
all_nodes += df_sim[act_index]['target'].unique().tolist()
unique_nodes = set(all_nodes)
for uuid_g in unique_nodes:
    uuid = uuid_g.replace('_', '-')
    uuid_index = df_sample['uuid'] == uuid
    label = df_sample[uuid_index]['label'].iloc[0]
    thumb_uri = df_sample[uuid_index]['thumbnail_uri'].iloc[0]
    if str(thumb_uri).startswith('http'):
        net.add_node(
            uuid_g, 
            label=label,
            url=f'https://opencontext.org/subjects/{uuid}',
            shape='image', 
            image=thumb_uri,
            size=25,
        )
    else:
        net.add_node(
            uuid_g, 
            label=label,
            url=f'https://opencontext.org/subjects/{uuid}',
            shape='dot', 
            size=20,
        )

# 4. Add edges
max_len = 100
min_len = 30
max_weight = df_sim[act_index]['weight'].max()
for _, row in df_sim[act_index].iterrows():
    value = (row['weight'] / max_weight) * 15
    if False and row['weight'] >= q_top:
        spring_length = max_len - (row['weight'] / max_weight)  * (max_len - min_len)
        if spring_length < min_len:
            spring_length = min_len
        if spring_length > max_len:
            spring_length = max_len
        net.add_edge(
            row['source'], 
            row['target'], 
            value=value,
            length=spring_length,
            weight=row['weight'],
            physics=(row['weight'] > 0.05),
        )
    else:
        net.add_edge(
            row['source'], 
            row['target'], 
            value=value,
            weight=row['weight'],
        )

# 5. Generate and open the graph
# net.toggle_physics(True)
net.prep_notebook()



In [25]:
# net.show(net_output_html, notebook=False)
net.save_graph(net_output_html)

# Add this AFTER calling net.write_html() or generating your file
with open(net_output_html, "r") as f:
    html = f.read()

# Inject JS event listener to freeze physics on load, add onclick event
add_script = """
network = new vis.Network(container, data, options);

// network.clusterByHubsize(4);

network.once("stabilized", function() {
    network.setOptions({ physics: false });
});
network.on("click", function(params) {
    if (params.nodes.length > 0) {
        var nodeId = params.nodes[0];
        var nodeData = nodes.get(nodeId);
        if (network.isCluster(nodeId)) {
            network.openCluster(nodeId);
        }
        if (nodeData && nodeData.url) {
            window.open(nodeData.url, '_blank'); // Opens in a new tab
        }
    }
});

"""
html = html.replace("network = new vis.Network(container, data, options);", add_script, 1)
html = html.replace('<meta charset="utf-8">', f'<meta charset="utf-8"><title>{TITLE}</title>', 1)
html = html.replace('<div id="mynetwork"', f'<h1>{TITLE}</h1> \n<div id="mynetwork"', 1)

with open(net_output_html, "w") as f:
    f.write(html)

from IPython.display import IFrame
from IPython.display import display, HTML

HTML(filename=net_output_html)
# 1. Write the standalone HTML file to disk
# net.write_html(net_output_html)

# 2. Force it to load manually inside a cell window
# IFrame(src=net_output_html, width="100%", height="750px")

# OLD CODE
from d3blocks import D3Blocks

d3 = D3Blocks()
d3.elasticgraph(df=df_sim[act_index], filepath=output_html, charge=2500)
for uuid_g, node_dict in d3.Elasticgraph.D3graph.node_properties.items():
    uuid = uuid_g.replace('_', '-')
    act_index = df_sample['uuid'] == uuid
    if df_sample[act_index].empty:
        print(f'Cannot update {uuid_g} from {uuid}')
        print(node_dict)
        continue
    item_class = str(df_sample[act_index]['item_class__slug'].iloc[0])
    item_class = item_class.split('-cat-')[-1]
    label = df_sample[act_index]['label'].iloc[0]
    # print(f'Group {item_class} label {label}')
    # d3.Elasticgraph.D3graph.node_properties[uuid_g]['group'] = item_class
    d3.Elasticgraph.D3graph.node_properties[uuid_g]['label'] = label
    d3.Elasticgraph.D3graph.node_properties[uuid_g]['fontsize'] = 6
    
for s_t_tup, edge_dict in d3.Elasticgraph.D3graph.edge_properties.items():
    source = s_t_tup[0]
    target = s_t_tup[1]
    weight = float(edge_dict['weight'])
    edge_w = round((3 * weight), 2)
    d3.Elasticgraph.D3graph.edge_properties[(source, target)]['edge_size'] = edge_w

html = d3.Elasticgraph.show(filepath=output_html)


